# Stage 05 - Readiness Drift Monitoring

Measure feature, prediction, and source-data drift so consumers can judge whether scores remain trustworthy.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILE = "readiness_observation_features.csv"
FEATURE_TABLE = "silver_readiness_feature_store"
DATA_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_FILE,
    Path("../data") / DATA_FILE,
    Path("data") / DATA_FILE,
    Path("Files") / DATA_FILE,
]


def locate_data_file(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {DATA_FILE}. Checked: {checked}")


def min_max_scale(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / span

spark_session = globals().get("spark")
if spark_session is not None and spark_session.catalog.tableExists(FEATURE_TABLE):
    frame = spark_session.table(FEATURE_TABLE).toPandas()
    data_path = f"Lakehouse table {FEATURE_TABLE}"
else:
    data_path = locate_data_file(DATA_CANDIDATES)
    frame = pd.read_csv(data_path)

frame["feature_timestamp_utc"] = pd.to_datetime(frame["feature_timestamp_utc"], utc=True)
frame = frame.sort_values(
    ["scenario_id", "simulation_run_id", "system_instance_id", "feature_timestamp_utc"]
).reset_index(drop=True)
frame["quality_gap"] = 1.0 - frame["quality_rate"]
frame["feature_recency_minutes"] = frame["track_freshness_seconds"] / 60.0
frame["health_decline_flag"] = (frame["health_trend_index"] < 0).astype(int)
frame["maintenance_age_band_index"] = frame["maintenance_age_category"].map(
    {"fresh-service": 0, "steady-cycle": 1, "extended-cycle": 2}
).astype(int)
frame["run_quality_delta"] = frame["quality_rate"] - frame.groupby("simulation_run_id")["quality_rate"].transform("mean")
gap_totals = frame.groupby("simulation_run_id")["gap_count"].transform("sum").replace(0, 1)
frame["run_gap_share"] = frame["gap_count"] / gap_totals
frame["baseline_alignment_gap"] = frame["baseline_deviation_index"] + frame["quality_gap"]
frame["deterministic_baseline_score"] = (
    0.30 * min_max_scale(frame["track_freshness_seconds"])
    + 0.20 * min_max_scale(frame["gap_count"])
    + 0.20 * min_max_scale(frame["quality_gap"])
    + 0.15 * min_max_scale(frame["abstract_ack_lag_seconds"])
    + 0.10 * min_max_scale(frame["baseline_deviation_index"])
    + 0.05 * min_max_scale(frame["maintenance_age_days"])
)

feature_columns = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'maintenance_age_category', 'test_phase', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
numeric_features = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
categorical_features = ["maintenance_age_category", "test_phase"]
run_order = frame.groupby("simulation_run_id")["feature_timestamp_utc"].min().sort_values().index.tolist()


In [ ]:
def calculate_psi(reference, current, bins=5):
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(reference, quantiles))
    if len(edges) < 3:
        return 0.0
    reference_counts, _ = np.histogram(reference, bins=edges)
    current_counts, _ = np.histogram(current, bins=edges)
    reference_ratio = np.clip(reference_counts / max(reference_counts.sum(), 1), 1e-6, None)
    current_ratio = np.clip(current_counts / max(current_counts.sum(), 1), 1e-6, None)
    return float(np.sum((current_ratio - reference_ratio) * np.log(current_ratio / reference_ratio)))


def demo_drift_band(psi_value):
    if psi_value < 0.10:
        return "demo-stable"
    if psi_value < 0.20:
        return "demo-watch"
    return "demo-review"


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            categorical_features,
        ),
    ]
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=7)),
    ]
)
model.fit(frame[feature_columns], frame["synthetic_review_priority_label"])
frame["model_priority_probability"] = model.predict_proba(frame[feature_columns])[:, 1]

reference_runs = run_order[:2]
current_runs = run_order[2:]
reference_frame = frame[frame["simulation_run_id"].isin(reference_runs)].copy()
current_frame = frame[frame["simulation_run_id"].isin(current_runs)].copy()

drift_features = [
    "track_freshness_seconds",
    "quality_rate",
    "abstract_ack_lag_seconds",
    "health_trend_index",
    "baseline_deviation_index",
    "model_priority_probability",
]
drift_rows = []
for feature_name in drift_features:
    psi_value = calculate_psi(reference_frame[feature_name], current_frame[feature_name])
    drift_rows.append(
        {
            "feature_name": feature_name,
            "reference_mean": round(float(reference_frame[feature_name].mean()), 4),
            "current_mean": round(float(current_frame[feature_name].mean()), 4),
            "psi": round(psi_value, 4),
            "demo_drift_band": demo_drift_band(psi_value),
        }
    )
drift_summary = pd.DataFrame(drift_rows).sort_values("psi", ascending=False).reset_index(drop=True)

drift_summary


In [ ]:
run_comparison = frame.groupby(["scenario_id", "simulation_run_id"], as_index=False).agg(
    observed_priority_rate=("synthetic_review_priority_label", "mean"),
    deterministic_baseline_rate=("deterministic_baseline_score", "mean"),
    predicted_priority_rate=("model_priority_probability", "mean"),
    avg_quality_rate=("quality_rate", "mean"),
    avg_baseline_deviation=("baseline_deviation_index", "mean"),
)

monitoring_summary = {
    "classification": "SYNTHETIC_UNCLASS",
    "reference_runs": reference_runs,
    "current_runs": current_runs,
    "demo_only_guidance": "demo-stable, demo-watch, and demo-review are invented bands for this lesson only.",
    "lineage": {
        "source_snapshot_ids": sorted(frame["source_snapshot_id"].unique().tolist()),
        "feature_snapshot_ids": sorted(frame["feature_snapshot_id"].unique().tolist()),
        "baseline_snapshot_ids": sorted(frame["baseline_snapshot_id"].unique().tolist()),
    },
    "limitations": [
        "The drift bands are transparent demo-only labels.",
        "Only four synthetic runs are available, so the drift view is directional rather than statistically complete.",
        "Any follow-up action remains an analyst choice outside this notebook.",
    ],
}

print(json.dumps(monitoring_summary, indent=2))
spark_session = globals().get("spark")
if spark_session is not None:
    spark_session.createDataFrame(drift_summary).write.mode("overwrite").saveAsTable("gold_readiness_drift_summary")
    print("Saved optional Lakehouse table: gold_readiness_drift_summary")
else:
    print("Spark session not detected. Skipping optional Lakehouse table write.")

run_comparison
